# 8장 Cox 모형과 비례위험 점검 (R과 파이썬으로 하는 통계분석)

survival 패키지의 원발성 담즙성 담관염 자료(pbc)로 Cox 모형을 맞추고, 비례위험 가정을 점검하고, 보정 생존곡선을 그립니다. 보건통계학 기초의 파이썬 실습 11과 같은 자료입니다. e 절에서는 보건통계학 기초 11장의 예를 계산합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요.


## a. 분석할 자료 만들기

### 셀 1. PBC 자료에서 무작위배정 환자 고르기

In [ ]:
library(survival)
d <- subset(pbc, !is.na(trt))      # 무작위배정된 312명만
nrow(d)
table(status = d$status, trt = d$trt)

### 셀 2. 사건, 시간, 공변량 만들기

In [ ]:
d$death  <- as.integer(d$status == 2)   # 사망 = 1, 간이식과 생존 = 0
d$years  <- d$time / 365.25
d$dpca   <- as.integer(d$trt == 1)      # D-penicillamine = 1, 위약 = 0
d$female <- as.integer(d$sex == "f")
d$age10  <- d$age / 10                  # 10세 단위
d$edema  <- factor(d$edema)             # 0, 0.5, 1을 범주로
colSums(is.na(d))[colSums(is.na(d)) > 0]

## b. Cox 모형과 위험비

### 셀 3. 치료만 넣은 Cox 모형

In [ ]:
fit1 <- coxph(Surv(years, death) ~ dpca, data = d)
summary(fit1)
survdiff(Surv(years, death) ~ dpca, data = d)$chisq

### 셀 4. 다변수 Cox 모형

In [ ]:
fit <- coxph(Surv(years, death) ~ dpca + age10 + female + log2(bili) +
               albumin + edema + protime, data = d)
summary(fit)

### 셀 5. 결측이 있는 변수를 넣으면

In [ ]:
coxph(Surv(years, death) ~ dpca + chol, data = d)

### 셀 6. 논문에 옮길 위험비 표

In [ ]:
library(broom)
hr <- tidy(fit, exponentiate = TRUE, conf.int = TRUE)
data.frame(term = hr$term,
           `HR (95% CI)` = sprintf("%.2f (%.2f-%.2f)", hr$estimate,
                                   hr$conf.low, hr$conf.high),
           P = format.pval(hr$p.value, digits = 2, eps = 0.001),
           check.names = FALSE)

### 셀 7. 빌리루빈을 어떤 꼴로 넣을지

In [ ]:
fit_lin <- update(fit, . ~ . - log2(bili) + bili)   # 빌리루빈을 그대로 넣은 모형
AIC(fit_lin, fit)

## c. 비례위험 가정 점검

### 셀 8. cox.zph()로 비례위험 검정

In [ ]:
zph <- cox.zph(fit)
zph

### 셀 9. 시간 변환과 가변수별 검정

In [ ]:
cox.zph(fit, transform = "rank", terms = FALSE)

### 셀 10. 척도화 Schoenfeld 잔차 그림

In [ ]:
par(mfrow = c(1, 2))
plot(zph, var = "dpca")
abline(h = coef(fit)["dpca"], lty = 3, col = "gray40")
plot(zph, var = "protime")
abline(h = coef(fit)["protime"], lty = 3, col = "gray40")

### 셀 11. 부종 범주별 log(-log) 그림

In [ ]:
km_ed <- survfit(Surv(years, death) ~ edema, data = d)
cols <- c("black", "#1f5f8b", "#c8553d")
plot(km_ed, fun = "cloglog", col = cols,
     xlab = "Years (log scale)", ylab = "log(-log S(t))")
legend("topleft", c("No edema", "Edema 0.5", "Edema 1.0"),
       col = cols, lty = 1, bty = "n")

## d. 가정이 어긋날 때와 보정 생존곡선

### 셀 12. 부종으로 층화한 Cox 모형

In [ ]:
fit_s <- coxph(Surv(years, death) ~ dpca + age10 + female + log2(bili) +
                 albumin + protime + strata(edema), data = d)
fit_s

### 셀 13. 추적 기간을 3년에서 나눈 긴 자료

In [ ]:
long <- survSplit(Surv(years, death) ~ ., data = d, cut = 3, episode = "period")
c(nrow(d), nrow(long))
long[long$id %in% c(1, 4), c("id", "tstart", "years", "death", "period", "protime")]

### 셀 14. 기간별 위험비

In [ ]:
long$late <- as.integer(long$period == 2)
fit_t <- coxph(Surv(tstart, years, death) ~ dpca + age10 + female + log2(bili) +
                 albumin + edema + protime + protime:late, data = long)
round(summary(fit_t)$coefficients[c("dpca", "protime", "protime:late"), ], 4)
exp(coef(fit_t)["protime"] + coef(fit_t)["protime:late"])   # 3년 뒤의 위험비

### 셀 15. 직접 표준화한 보정 생존곡선

In [ ]:
std_surv <- function(newd, times)      # 환자마다 예측한 곡선의 평균
  rowMeans(summary(survfit(fit, newdata = newd), times = times)$surv)
at <- c(1, 5, 10)
round(cbind(years   = at,
            dpca    = std_surv(transform(d, dpca = 1), at),
            placebo = std_surv(transform(d, dpca = 0), at)), 3)
grid <- seq(0, 12, by = 0.1)
plot(grid, std_surv(transform(d, dpca = 1), grid), type = "s", ylim = c(0, 1),
     col = "#1f5f8b", xlab = "Years since randomization",
     ylab = "Adjusted survival probability")
lines(grid, std_surv(transform(d, dpca = 0), grid), type = "s", col = "#c8553d")
legend("bottomleft", c("D-penicillamine", "Placebo"),
       col = c("#1f5f8b", "#c8553d"), lty = 1, bty = "n")

## e. 보건통계학 기초 11장의 예

### 셀 16. 약물만 넣은 Cox 모형 (11장 '위험률과 위험비')

In [ ]:
co <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch11_cohort.csv")
co$drug      <- factor(co$drug, levels = c("B", "A"))            # 약물 B가 기준
co$imdc      <- factor(co$imdc, levels = c("favorable", "intermediate", "poor"))
co$histology <- factor(co$histology, levels = c("clear", "nonclear"))
f1 <- coxph(Surv(time, status) ~ drug, data = co)
summary(f1)
f1$loglik                     # 계수가 모두 0일 때와 추정값에서의 로그 부분우도

### 셀 17. 동점 처리 방법과 위험비 (표 11-3)

In [ ]:
ties <- c("efron", "breslow", "exact")
hr_by <- function(tm) sapply(ties, function(m)
  exp(coef(coxph(Surv(tm, status) ~ drug, data = co, ties = m))))
co$t6 <- ceiling(co$time / 6) * 6          # 사망 시간을 6개월 단위로 뭉뚱그림
round(rbind(original = hr_by(co$time), by_6_months = hr_by(co$t6)), 4)

### 셀 18. 다변수 모형 (11장 '논문에서 읽기'의 Table 2)

In [ ]:
fm <- coxph(Surv(time, status) ~ drug + I(age / 10) + male + imdc +
              nephrectomy + histology, data = co)
round(cbind(HR = exp(coef(fm)), exp(confint(fm)),
            P = summary(fm)$coefficients[, "Pr(>|z|)"]), 3)
anova(update(fm, . ~ . - imdc), fm)        # IMDC 위험군 전체(자유도 2)의 우도비 검정
fm_i <- update(fm, . ~ . - imdc + relevel(imdc, ref = "intermediate"))
round(exp(cbind(coef(fm_i), confint(fm_i)))[7, ], 2)   # poor 대 intermediate

### 셀 19. 비례위험 검정과 시간 변환

In [ ]:
cox.zph(fm)                                       # 기본 transform = "km"
rbind(identity = cox.zph(fm, transform = "identity")$table[c("male", "histology", "GLOBAL"), "p"],
      rank     = cox.zph(fm, transform = "rank")$table[c("male", "histology", "GLOBAL"), "p"],
      log      = cox.zph(fm, transform = "log")$table[c("male", "histology", "GLOBAL"), "p"])

### 셀 20. 기간별 위험비, 층화, 시간과의 교호작용

In [ ]:
long <- survSplit(Surv(time, status) ~ ., data = co, cut = 12, episode = "period")
long$nc_0_12  <- as.integer(long$histology == "nonclear" & long$period == 1)
long$nc_after <- as.integer(long$histology == "nonclear" & long$period == 2)
fp <- coxph(Surv(tstart, time, status) ~ drug + I(age / 10) + male + imdc +
              nephrectomy + nc_0_12 + nc_after, data = long)
round(exp(cbind(HR = coef(fp), confint(fp)))[c("drugA", "nc_0_12", "nc_after"), ], 2)
anova(update(fp, . ~ . - nc_0_12 - nc_after + I(nc_0_12 + nc_after)), fp)   # 두 기간의 HR이 같은가
fs <- update(fm, . ~ . - histology + strata(histology))            # 조직형으로 층화
round(exp(cbind(coef(fs), confint(fs)))["drugA", ], 2)
round(cox.zph(fs)$table[, "p"], 3)                                  # 층화 뒤 나머지 변수의 비례위험 검정
co$nonclear <- as.integer(co$histology == "nonclear")
ft <- coxph(Surv(time, status) ~ drug + I(age / 10) + male + imdc + nephrectomy +
              histology + tt(nonclear), data = co,
            tt = function(x, t, ...) x * log(t))                   # 조직형 × log(개월)
round(summary(ft)$coefficients[c("histologynonclear", "tt(nonclear)"), 1:3], 4)

### 셀 21. 직접 표준화한 보정 생존율과 한 환자의 예측

In [ ]:
std_surv <- function(g, times) {          # 모두 약물 g를 받았다고 놓은 곡선의 평균
  nd <- transform(co, drug = factor(g, levels = c("B", "A")))
  rowMeans(summary(survfit(fm, newdata = nd), times = times)$surv)
}
at <- c(12, 24, 48)
round(rbind(A = std_surv("A", at), B = std_surv("B", at)), 3)
summary(survfit(fm, newdata = data.frame(drug = c("B", "A"), age = 70, male = 1,
  imdc = "intermediate", nephrectomy = 1, histology = "clear")), times = c(12, 24))$surv

### 셀 22. 역확률 가중 Cox 모형

In [ ]:
ps <- glm(drug == "A" ~ I(age / 10) + male + imdc + nephrectomy + histology,
          family = binomial, data = co)$fitted.values      # 약물 A를 받을 확률
pA <- mean(co$drug == "A")
co$w_s <- ifelse(co$drug == "A", pA / ps, (1 - pA) / (1 - ps))   # 안정화 가중치
co$w_u <- ifelse(co$drug == "A", 1 / ps, 1 / (1 - ps))           # 안정화하지 않은 가중치
round(range(co$w_s), 2)
fw <- coxph(Surv(time, status) ~ drug, data = co, weights = w_s)
summary(fw)$coefficients
round(summary(fw)$conf.int, 3)
fu <- coxph(Surv(time, status) ~ drug, data = co, weights = w_u)
round(summary(fu)$coefficients[, c("se(coef)", "robust se")], 4)

### 셀 23. 강건 표준오차와 한 명씩 빼고 다시 적합하기

In [ ]:
fr <- update(fm, robust = TRUE)           # 환자 한 명이 한 줄이면 id가 군집
round(summary(fr)$coefficients["drugA", c("se(coef)", "robust se")], 4)
loo <- sapply(seq_len(nrow(co)), function(i)         # 한 명씩 빼고 다시 적합
  coef(update(fm, data = co[-i, ]))["drugA"])
round(exp(range(loo)), 3)
se <- sqrt(fm$var[1, 1])
round(max(abs(coef(fm)["drugA"] - loo)) / se, 2)              # 가장 큰 변화 ÷ 표준오차
round(max(abs(residuals(fm, type = "dfbeta")[, 1])) / se, 2)  # 다시 적합하지 않은 근사

### 셀 24. 혈압을 스플라인으로 넣기 (그림 11-2)

In [ ]:
bp <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch11_bp.csv")
kn <- quantile(bp$sbp, c(0.05, 0.35, 0.65, 0.95))    # 매듭: 5·35·65·95번째 백분위수
round(kn, 1)
rcs4 <- function(x) Hmisc::rcspline.eval(x, knots = kn, inclx = TRUE)   # 제한 삼차 스플라인
f_lin <- coxph(Surv(time, status) ~ I(sbp / 10), data = bp)
f_rcs <- coxph(Surv(time, status) ~ rcs4(sbp), data = bp)
round(exp(coef(f_lin)), 3)                            # 10 mmHg당 위험비(직선)
anova(f_lin, f_rcs)                                   # 곡선이 필요한가(자유도 2)
at <- c(100, 140, 160, 180)
X  <- sweep(rcs4(at), 2, rcs4(120))                   # 120 mmHg와의 차이
lp <- drop(X %*% coef(f_rcs)); se <- sqrt(diag(X %*% vcov(f_rcs) %*% t(X)))
round(cbind(sbp = at, HR = exp(lp), lower = exp(lp - 1.96 * se), upper = exp(lp + 1.96 * se)), 2)

## f. 과제 정답

### 셀 25. 과제 1 정답: 빌리루빈과 나이의 위험비

In [ ]:
b <- coef(fit)
exp(b["log2(bili)"] * log2(c(2 / 1, 10 / 5, 11 / 10)))   # 1→2, 5→10, 10→11 mg/dL
exp(2 * b["age10"])                                         # 20세 많을 때

### 셀 26. 과제 2 정답: 간이식도 사건으로 세면

In [ ]:
fit_tx <- update(fit, Surv(years, status > 0) ~ .)    # 사망 또는 간이식
c(death = fit$nevent, death_or_tx = fit_tx$nevent)
round(cbind(death = exp(coef(fit)), death_or_tx = exp(coef(fit_tx))), 3)

### 셀 27. 과제 3 정답: 치료 × 추적 시간

In [ ]:
wrong <- coxph(Surv(years, death) ~ dpca + age10 + female + log2(bili) +
                 albumin + edema + protime + dpca:years, data = d)
round(summary(wrong)$coefficients[c("dpca", "dpca:years"), ], 4)
right <- coxph(Surv(years, death) ~ dpca + age10 + female + log2(bili) +
                 albumin + edema + protime + tt(dpca), data = d,
               tt = function(x, t, ...) x * t)
round(summary(right)$coefficients[c("dpca", "tt(dpca)"), ], 4)